# Band-split VAE Training (Colab)
Mount Drive, set paths, and train the band-split VAE on lip landmarks.

In [ ]:

import os
import sys
from pathlib import Path

use_colab = 'google.colab' in sys.modules
if use_colab:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    base_dir = Path('/content/drive/MyDrive/liveness_detection_vae')
else:
    base_dir = Path.cwd()

project_dir = base_dir
data_dir = project_dir / '20GBprocessed'
save_dir = project_dir / 'runs' / 'bandvae_colab'
save_dir.mkdir(parents=True, exist_ok=True)

os.chdir(project_dir)
print(f'Project dir: {project_dir}')
print(f'Data dir: {data_dir}')
print(f'Save dir: {save_dir}')


In [ ]:

import time
import torch
import torch.optim as optim
from torch.utils.data import DataLoader, random_split

from config_bandvae import get_config
from dataset_bandvae import LipLivenessBandDataset
from model_bandvae import BandSplitVAE, band_split_vae_loss

device = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')

config = get_config('simple')
config.data_dir = str(data_dir)
config.save_dir = str(save_dir)
config.device = device
config.num_workers = 4
print(config)


In [ ]:

dataset = LipLivenessBandDataset(
    data_dir=config.data_dir,
    T_fixed=config.T_fixed,
    fps=config.fps,
    use_procrustes=True,
    use_acceleration=config.use_acceleration,
    use_angle=config.use_angle,
    use_angle_rate=config.use_angle_rate,
    fc_low=config.fc_low,
    fc_high=config.fc_high,
    filter_order=config.filter_order,
)

train_size = int((1 - config.val_split) * len(dataset))
val_size = len(dataset) - train_size
train_set, val_set = random_split(
    dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(
    train_set,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=config.num_workers,
    pin_memory=True,
)

val_loader = DataLoader(
    val_set,
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=config.num_workers,
    pin_memory=True,
)

print(f'Train: {len(train_set)}, Val: {len(val_set)}')


In [ ]:

model = BandSplitVAE(
    C_in_per_band=config.C_in_per_band,
    C_h=config.C_h,
    C_z=config.C_z,
    dilations=config.dilations,
).to(config.device)

optimizer = optim.Adam(model.parameters(), lr=config.lr)


In [ ]:

def train_epoch(model, loader, optimizer, config, epoch):
    model.train()
    total_loss = recon_lf = recon_bp = recon_hf = 0.0
    kl_lf = kl_bp = kl_hf = 0.0
    n_batches = len(loader)
    start_time = time.time()

    for batch_idx, (x_lf, x_bp, x_hf) in enumerate(loader):
        x_lf = x_lf.to(config.device)
        x_bp = x_bp.to(config.device)
        x_hf = x_hf.to(config.device)

        recons, mus, logvars, x_hat_fused = model(x_lf, x_bp, x_hf)
        targets = {'lf': x_lf, 'bp': x_bp, 'hf': x_hf}
        betas = {'lf': config.beta_lf, 'bp': config.beta_bp, 'hf': config.beta_hf}

        loss, loss_dict = band_split_vae_loss(
            recons, mus, logvars, targets, x_hat_fused, None, betas=betas, alpha_fusion=0.0
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss_dict['total']
        recon_lf += loss_dict['recon_lf']
        recon_bp += loss_dict['recon_bp']
        recon_hf += loss_dict['recon_hf']
        kl_lf += loss_dict['kl_lf']
        kl_bp += loss_dict['kl_bp']
        kl_hf += loss_dict['kl_hf']

        if (batch_idx + 1) % 50 == 0 or (batch_idx + 1) == n_batches:
            elapsed = time.time() - start_time
            avg_time = elapsed / (batch_idx + 1)
            eta = (n_batches - (batch_idx + 1)) * avg_time
            n = batch_idx + 1
            total_recon = (recon_lf + recon_bp + recon_hf) / n
            total_kl = (kl_lf + kl_bp + kl_hf) / n
            print(
                f"[Epoch {epoch:02d} | {n:04d}/{n_batches}] "
                f"Total={total_loss / n:.4f} (Recon={total_recon:.4f} KL={total_kl:.4f}) "
                f"LF[R={recon_lf / n:.4f} K={kl_lf / n:.4f}] "
                f"BP[R={recon_bp / n:.4f} K={kl_bp / n:.4f}] "
                f"HF[R={recon_hf / n:.4f} K={kl_hf / n:.4f}] ETA={eta/60:.1f}m"
            )

    return {
        'total': total_loss / n_batches,
        'recon_lf': recon_lf / n_batches,
        'recon_bp': recon_bp / n_batches,
        'recon_hf': recon_hf / n_batches,
        'kl_lf': kl_lf / n_batches,
        'kl_bp': kl_bp / n_batches,
        'kl_hf': kl_hf / n_batches,
    }


def validate(model, loader, config):
    model.eval()
    total_loss = recon_lf = recon_bp = recon_hf = 0.0
    kl_lf = kl_bp = kl_hf = 0.0
    n_samples = 0

    with torch.no_grad():
        for x_lf, x_bp, x_hf in loader:
            x_lf = x_lf.to(config.device)
            x_bp = x_bp.to(config.device)
            x_hf = x_hf.to(config.device)

            recons, mus, logvars, x_hat_fused = model(x_lf, x_bp, x_hf)
            targets = {'lf': x_lf, 'bp': x_bp, 'hf': x_hf}
            betas = {'lf': config.beta_lf, 'bp': config.beta_bp, 'hf': config.beta_hf}

            loss, loss_dict = band_split_vae_loss(
                recons, mus, logvars, targets, x_hat_fused, None, betas=betas, alpha_fusion=0.0
            )

            batch_size = x_lf.size(0)
            total_loss += loss_dict['total'] * batch_size
            recon_lf += loss_dict['recon_lf'] * batch_size
            recon_bp += loss_dict['recon_bp'] * batch_size
            recon_hf += loss_dict['recon_hf'] * batch_size
            kl_lf += loss_dict['kl_lf'] * batch_size
            kl_bp += loss_dict['kl_bp'] * batch_size
            kl_hf += loss_dict['kl_hf'] * batch_size
            n_samples += batch_size

    return {
        'total': total_loss / n_samples,
        'recon_lf': recon_lf / n_samples,
        'recon_bp': recon_bp / n_samples,
        'recon_hf': recon_hf / n_samples,
        'kl_lf': kl_lf / n_samples,
        'kl_bp': kl_bp / n_samples,
        'kl_hf': kl_hf / n_samples,
    }


In [ ]:

best_val = float('inf')
best_path = save_dir / 'best.pt'

for epoch in range(1, config.epochs + 1):
    print(f'
Epoch {epoch}/{config.epochs}')
    train_metrics = train_epoch(model, train_loader, optimizer, config, epoch)
    val_metrics = validate(model, val_loader, config)

    train_recon = train_metrics['recon_lf'] + train_metrics['recon_bp'] + train_metrics['recon_hf']
    train_kl = train_metrics['kl_lf'] + train_metrics['kl_bp'] + train_metrics['kl_hf']
    val_recon = val_metrics['recon_lf'] + val_metrics['recon_bp'] + val_metrics['recon_hf']
    val_kl = val_metrics['kl_lf'] + val_metrics['kl_bp'] + val_metrics['kl_hf']

    print(
        f"Train Total={train_metrics['total']:.4f} Recon={train_recon:.4f} KL={train_kl:.4f} | "
        f"Val Total={val_metrics['total']:.4f} Recon={val_recon:.4f} KL={val_kl:.4f}"
    )

    if val_metrics['total'] < best_val:
        best_val = val_metrics['total']
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_metrics': val_metrics,
            'config': config,
        }, best_path)
        print(f'Saved new best to {best_path}')
